# **Neuro-ATS**

In [1]:
from dotenv import load_dotenv
import os
load_dotenv()

os.environ["LLAMA_CLOUD_API_KEY"] = os.getenv("LLAMA_CLOUD_API_KEY")

In [2]:
from llama_cloud_services import LlamaParse

parser = LlamaParse(
    num_workers=2,
    verbose=True,
    language="en"
)

In [3]:
result = await parser.aparse("data/myCV-16th.pdf")

Started parsing the file under job_id f3763e34-b124-4f09-b7f4-bdea1d6b22f2


In [ ]:
# get the llama-index markdown documents
markdown_documents = result.get_markdown_documents(split_by_page=True)

# get the llama-index text documents
text_documents = result.get_text_documents(split_by_page=False)



# Items will vary based on the parser configuration
for page in result.pages:
    print(page.text)
    print(page.md)
    print(page.images)
    print(page.layout)
    print(page.structuredData)

Tahmid Zamee
 # tahmidzamee1999@gmail.com          ÷ Tahmid Zamee  æ Tahmid Zamee
 +880-1601751654

 Objective

An ambitious and passionate personality, seeking for a role where I can apply my knowledge in
programming, problem-solving and analytical skills.

 Education

2020 – 2025   ] B.Sc, Computer Science and Engineering, Independent University, Bangladesh

 Research Publications

 Journal Paper

 1     S. Hafiz, T. Zamee, and M. A. Rahman, “Agrotrust: A blockchain-enabled ai-powered organic product
       certification system,” in 4th World Conference on Information Systems for Business Management (ISBM
       2025), Bangkok, Thailand, 2025.

 Skills

Programming Languages         ]  Java, Python, PHP
             Databases        ]  MySQL
             Web Dev          ]  HTML, CSS, JavaScript
             Languages        ]  Bangla - Native, English - Fluent.

 Projects

      ]  Market Information System for Farmers – Developed a web-based platform to provide farm-
         ers w

In [13]:
result.get_text

<bound method JobResult.get_text of JobResult(pages=[Page(page=1, text='Tahmid Zamee\n # tahmidzamee1999@gmail.com          ÷ Tahmid Zamee  æ Tahmid Zamee\n +880-1601751654\n\n Objective\n\nAn ambitious and passionate personality, seeking for a role where I can apply my knowledge in\nprogramming, problem-solving and analytical skills.\n\n Education\n\n2020 – 2025   ] B.Sc, Computer Science and Engineering, Independent University, Bangladesh\n\n Research Publications\n\n Journal Paper\n\n 1     S. Hafiz, T. Zamee, and M. A. Rahman, “Agrotrust: A blockchain-enabled ai-powered organic product\n       certification system,” in 4th World Conference on Information Systems for Business Management (ISBM\n       2025), Bangkok, Thailand, 2025.\n\n Skills\n\nProgramming Languages         ]  Java, Python, PHP\n             Databases        ]  MySQL\n             Web Dev          ]  HTML, CSS, JavaScript\n             Languages        ]  Bangla - Native, English - Fluent.\n\n Projects\n\n      ]  

## **Professional Documents Parsing**

In [14]:
import nest_asyncio
nest_asyncio.apply()

import os
from dotenv import load_dotenv
from llama_cloud_services import LlamaParse
import asyncio

load_dotenv()

# --- CONFIGURATION ---
# The instruction is CRITICAL. It tells LlamaParse how to interpret the layout.
RESUME_PARSING_INSTRUCTION = """
This is a professional resume. 
Preserve the structure of the document.
Extract the following clearly:
1. Contact Information
2. Professional Summary
3. Skills (Grouped by category if possible)
4. Work Experience (Company, Role, Dates, Responsibilities)
5. Education
Do not hallucinate information not present in the document.
"""

async def parse_resume_pro(file_path):
    """
    Robust function to parse a single resume with error handling.
    """
    try:
        print(f"⏳ Starting parse: {file_path}")
        
        # Initialize parser with specific Resume instructions
        parser = LlamaParse(
            api_key=os.getenv("LLAMA_CLOUD_API_KEY"),
            result_type="markdown",  # Markdown is best for LLM processing later
            parsing_instruction=RESUME_PARSING_INSTRUCTION, 
            verbose=False,
            language="en",
            num_workers=4 # Parallel workers for faster processing
        )

        # Async load enables concurrency
        documents = await parser.aload_data(file_path)
        
        if not documents:
            print(f"❌ Warning: No content extracted from {file_path}")
            return None

        # Combine pages into one markdown string
        full_markdown = "\n\n".join([doc.text for doc in documents])
        
        print(f"✅ Successfully parsed: {file_path}")
        return full_markdown

    except Exception as e:
        print(f"🚨 Error parsing {file_path}: {str(e)}")
        # In production, log this to a file or monitoring system (LangSmith)
        return None

# --- BATCH PROCESSING (Simulating 500 resumes) ---
async def process_batch(file_paths):
    tasks = [parse_resume_pro(path) for path in file_paths]
    # Gather results concurrently
    results = await asyncio.gather(*tasks)
    return results



In [15]:
# --- MAIN EXECUTION ---
if __name__ == "__main__":
    # Example list of resume paths (In real life, get this from your folder)
    resume_files = ["data/myCV-16th.pdf", "data/Nahid_Hasan_Riad_Resume_Python.pdf"] 
    
    # Run the async loop
    parsed_resumes = asyncio.run(process_batch(resume_files))
    
    # Check results
    for i, content in enumerate(parsed_resumes):
        if content:
            print(f"\n--- Resume {i+1} Output Preview ---\n")
            print(content[:500]) # Print first 500 chars only

⏳ Starting parse: data/myCV-16th.pdf
⏳ Starting parse: data/Nahid_Hasan_Riad_Resume_Python.pdf
✅ Successfully parsed: data/myCV-16th.pdf
✅ Successfully parsed: data/Nahid_Hasan_Riad_Resume_Python.pdf

--- Resume 1 Output Preview ---

1. **Contact Information**
- Name: Tahmid Zamee
- Email: tahmidzamee1999@gmail.com
- Phone: +880-1601751654

2. **Professional Summary**
- An ambitious and passionate personality, seeking for a role where I can apply my knowledge in programming, problem-solving and analytical skills.

3. **Skills**
- **Programming Languages:** Java, Python, PHP
- **Databases:** MySQL
- **Web Development:** HTML, CSS, JavaScript
- **Languages:** Bangla - Native, English - Fluent

4. **Work Experience**
- (No spec

--- Resume 2 Output Preview ---

1. **Contact Information**
- Name: Nahid Hasan Riad
- Location: Dhaka, Bangladesh
- Email: nhriyaad@gmail.com
- Phone: +8801840422522
- LinkedIn: linkedin.com/in/nhriyaad/
- GitHub: github.com/nahidn4p
- Portfolio: nhriyaad.pythona

In [16]:
resume_files = ["data/saiful islam mohim.pdf"] 

parsed_resumes = asyncio.run(process_batch(resume_files))

⏳ Starting parse: data/saiful islam mohim.pdf
✅ Successfully parsed: data/saiful islam mohim.pdf


In [17]:
for i, content in enumerate(parsed_resumes):
        if content:
            print(f"\n--- Resume {i+1} Output Preview ---\n")
            print(content) # Print first 500 chars only


--- Resume 1 Output Preview ---

1. **Contact Information**
- Name: Saiful Islam Mohim
- Email: saifulislammohim9@gmail.com
- Phone: +880 1783892863
- Address: Kuril Bissoroad, Mridha Bari, Vatara, Dhaka
- LinkedIn: [LinkedIn Profile]

2. **Professional Summary**
- (Not explicitly provided in the document)

3. **Skills**
- **Digital Marketing**
- Social media management
- Content creation
- SEO optimization
- **Analytical Skills**
- Proficient in market research
- Performance analysis
- **Tools**
- MS Word
- MS PowerPoint
- MS Excel
- Canva
- Basic knowledge of marketing platforms
- **Language Proficiency**
- Fluent in Bengali (Native)
- English

4. **Work Experience**
- **Company:** North South University Social Services Club
- **Role:** Executive Body
- **Dates:** 2021-2023
- **Responsibilities:**
- Organized and managed blood donation drives, winter clothing distribution initiatives, flood relief efforts, and the Onushongo program in collaboration with club members.
- Worked as par

In [3]:
import nest_asyncio
nest_asyncio.apply()

import os
from dotenv import load_dotenv
from llama_cloud_services import LlamaParse
import asyncio

load_dotenv()

async def parse_resume_to_markdown(file_path):
    """
    Parses PDF to Raw Markdown (Preserving Tables and Layout).
    This is the Input for your Extraction Agent.
    """
    try:
        parser = LlamaParse(
            api_key=os.getenv("LLAMA_CLOUD_API_KEY"),
            result_type="markdown",  # <--- This gives you Output B
            verbose=False,
            language="en",
            num_workers=2,
            # We REMOVED the "parsing_instruction" to get the raw, faithful data
        )

        # Async processing
        documents = await parser.aload_data(file_path)
        
        if not documents:
            return None

        # Combine pages (if resume is multi-page)
        full_markdown = "\n\n".join([doc.text for doc in documents])
        return full_markdown

    except Exception as e:
        print(f"Error parsing {file_path}: {e}")
        return None

# --- TEST IT ---
if __name__ == "__main__":
    # Replace with your actual file path
    resume_path = "data/saiful islam mohim.pdf" 
    
    # Run
    raw_markdown = asyncio.run(parse_resume_to_markdown(resume_path))
    
    print("--- RAW MARKDOWN FOR LLM AGENT ---")
    print(raw_markdown)

--- RAW MARKDOWN FOR LLM AGENT ---

Saiful Islam Mohim

Email: saifulislammohim9@gmail.com

Phone: +880 1783892863

Address: Kuril Bissoroad, Mridha Bari, Vatara, Dhaka

# Education

North South University, Dhaka, Bangladesh
2020-Current
Bachelor of Business Administration (Marketing)
Current CGPA: 2.44

Amrita Lal Dey College, Barisal
2018-2019
Higher Secondary School Certificate
GPA: 4.25

# Completed Marketing Courses

| MKT465: Brand Management                   | Grade: B+ |
| ------------------------------------------ | --------- |
| MKT337: Integrated Marketing Communication | Grade: B+ |
| MGT368: Entrepreneurship                   | Grade: B  |
| MKT330: Digital Marketing                  | Grade: B  |
| MKT344: Consumer Behaviour                 | Grade: B+ |
| MKT470: Marketing Research                 | Grade: B+ |

# Core Skills

- Digital Marketing: Social media management, content creation, and SEO optimization.
- Analytical Skills: Proficient in market research and perf

## **Start to build Agent via `LangGraph`**

In [1]:
from langgraph.graph import StateGraph, MessagesState
from pydantic import BaseModel, Field
from typing import List, Annotated

## **Rubric Scoring**

- if score < 40 --> Reject
- if score 40 - 70 --> Hr Reviews
- if score > 70 --> task assign